# 1st Place - March 2026 Kaggle Playground - Churn Prediction
This is my final submission to Kaggle's March 2026 Kaggle Playground Churn Prediction competition. Discussion writeup is [here][7]. GPT, Gemini, and Claude wrote 600k lines of code and trained 850 models. About 150 models were selected with forward feature selection and combined using `Nvidia cuML Logistic Regression`.

All OOF and Test PREDS in the code below come from levels 2 and 3. When a name has a suffix of `"_stk"` then it is level 3. Otherwise it is a level 2 model.

![](https://raw.githubusercontent.com/cdeotte/Kaggle_Images/refs/heads/main/Mar-2026/4LevelStack.png)

# KGMON Playbook for Tabular Data
This 1st place solution follows the KGMON Playbook 2026 for tabular data: Blog [here][1], Slides [here][2], GitHub [here][3], GTC Training Lab [here][4], KGMON homepage [here][5], Nvidia cuDF cuML homepage [here][6].
* EDA
* Build Baselines
* GPU Feature Engineering
* GPU Hill Climbing
* Stacking
* Pseudo Labeling
* GPU Extra Training

[1]: https://developer.nvidia.com/blog/the-kaggle-grandmasters-playbook-7-battle-tested-modeling-techniques-for-tabular-data/
[2]: https://docs.google.com/presentation/d/1857Vj4sg3LYiqU7fp9p-KNd-krunP6yZ/edit?usp=sharing&ouid=106005477169277841115&rtpof=true&sd=true
[3]: https://github.com/cdeotte/KGMON-Playbook-2026
[4]: https://www.nvidia.com/en-us/on-demand/session/gtc26-dlit81565/
[5]: https://www.nvidia.com/en-us/ai-data-science/kaggle-grandmasters/
[6]: https://docs.rapids.ai/install/
[7]: https://www.kaggle.com/competitions/playground-series-s6e3/writeups/1st-place-gpt5-4-gemini3-1-claudeopus4-6-kgm

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = "v1"

In [ ]:
# ============================================================
# LOGIT STACKING ENSEMBLE — cuML Logistic Regression
#   - 154 base models (OOF + test predictions)
#   - Fold-wise OOF calibration before stacking
#   - Final weights from full-data logistic regression fit
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata

try:
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
except Exception as e:
    raise ImportError(
        "Could not import cuML LogisticRegression. "
        "Make sure RAPIDS/cuML is installed in this environment."
    ) from e

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH = "/kaggle/input/competitions/playground-series-s6e3"
NUMPY_PATH = "/kaggle/input/datasets/cdeotte/s6e3-oof-and-test-pred-v2"
TRAIN_CSV  = f"{CSV_PATH}/train.csv"
TEST_CSV   = f"{CSV_PATH}/test.csv"
TARGET_COL = "Churn"
ID_COL     = "id"

N_SPLITS     = 5
RANDOM_STATE = 42
FOLD_CALIBRATE = True   # fold-wise OOF calibration before stacking

META_C       = 1e-2
META_SOLVER  = "qn"
META_PENALTY = "l2"
META_MAX_ITER = 10_000
FIT_INTERCEPT = True
META_TOL     = 1e-4

EPS        = 1e-15
LOGIT_CLIP = 30.0

SUBMISSION_PATH = f"submission_logitstack_{VER}.csv"
WEIGHTS_PATH    = f"ensemble_weights_logitstack_{VER}.csv"
OOF_PATH        = f"oof_logitstack_{VER}.npy"
TEST_PATH       = f"pred_logitstack_{VER}.npy"

# -----------------------------
# MODEL FILES  (name, oof_file, pred_file)
# All files are in the same directory as this notebook.
# -----------------------------
FILES = [
    ("xgb_v104", "oof_xgb_v104.npy", "pred_xgb_v104.npy"),
    ("logreg_v200", "oof_logreg_v200.npy", "pred_logreg_v200.npy"),
    ("logreg_v201", "oof_logreg_v201.npy", "pred_logreg_v201.npy"),
    ("nn_v302", "oof_nn_v302.npy", "pred_nn_v302.npy"),
    ("nn_v303", "oof_nn_v303.npy", "pred_nn_v303.npy"),
    ("nn_v304", "oof_nn_v304.npy", "pred_nn_v304.npy"),
    ("nn_v308", "oof_nn_v308.npy", "pred_nn_v308.npy"),
    ("nn_v309", "oof_nn_v309.npy", "pred_nn_v309.npy"),
    ("nn_v310", "oof_nn_v310.npy", "pred_nn_v310.npy"),
    ("nn_v312", "oof_nn_v312.npy", "pred_nn_v312.npy"),
    ("xgb_v400", "oof_xgb_v400.npy", "pred_xgb_v400.npy"),
    ("cat_v501", "oof_cat_v501.npy", "pred_cat_v501.npy"),
    ("cat_v502", "oof_cat_v502.npy", "pred_cat_v502.npy"),
    ("realmlp_v600", "oof_realmlp_telco_v600.npy", "pred_realmlp_telco_v600.npy"),
    ("lgbm_v1000", "oof_lgbm_v1000.npy", "pred_lgbm_v1000.npy"),
    ("xgb_v1201", "oof_xgb_v1201.npy", "pred_xgb_v1201.npy"),
    ("xgb_v1202", "oof_xgb_v1202.npy", "pred_xgb_v1202.npy"),
    ("xgb_v1203", "oof_xgb_v1203.npy", "pred_xgb_v1203.npy"),
    ("xgb_v1204", "oof_xgb_v1204.npy", "pred_xgb_v1204.npy"),
    ("xgb_v1300", "oof_xgb_v1300.npy", "pred_xgb_v1300.npy"),
    ("xgb_v1400", "oof_xgb_v1400.npy", "pred_xgb_v1400.npy"),
    ("xgb_v1500", "oof_xgb_v1500.npy", "pred_xgb_v1500.npy"),
    ("xgb_v1501", "oof_xgb_v1501.npy", "pred_xgb_v1501.npy"),
    ("xgb_v1502", "oof_xgb_v1502.npy", "pred_xgb_v1502.npy"),
    ("nn_v1600", "oof_nn_v1600.npy", "pred_nn_v1600.npy"),
    ("nn_v1602", "oof_nn_v1602.npy", "pred_nn_v1602.npy"),
    ("nn_v1603", "oof_nn_v1603.npy", "pred_nn_v1603.npy"),
    ("realmlp_v1700", "oof_realmlp_telco_v1700.npy", "pred_realmlp_telco_v1700.npy"),
    ("gnn_v1901", "oof_gnn_v1901.npy", "pred_gnn_v1901.npy"),
    ("gnn_v1906", "oof_gnn_v1906.npy", "pred_gnn_v1906.npy"),
    ("gnn_v1908", "oof_gnn_v1908.npy", "pred_gnn_v1908.npy"),
    ("gnn_v1909", "oof_gnn_v1909.npy", "pred_gnn_v1909.npy"),
    ("xgb_v2000", "oof_xgb_v2000.npy", "pred_xgb_v2000.npy"),
    ("cat_v2100", "oof_cat_v2100.npy", "pred_cat_v2100.npy"),
    ("xgb_v2200", "oof_xgb_v2200.npy", "pred_xgb_v2200.npy"),
    ("realmlp_v2300", "oof_realmlp_telco_v2300.npy", "pred_realmlp_telco_v2300.npy"),
    ("realmlp_v2400", "oof_realmlp_v2400.npy", "pred_realmlp_v2400.npy"),
    ("nn_v2500", "oof_nn_v2500.npy", "pred_nn_v2500.npy"),
    ("nn_v2600", "oof_nn_v2600.npy", "pred_nn_v2600.npy"),
    ("lgbm_v2700", "oof_lgbm_v2700.npy", "pred_lgbm_v2700.npy"),
    ("ftt_v2801", "oof_ftt_v2801.npy", "pred_ftt_v2801.npy"),
    ("gandalf_v1", "oof_gandalf_v1.npy", "pred_gandalf_v1.npy"),
    ("lgbm_v2900", "oof_lgbm_v2900.npy", "pred_lgbm_v2900.npy"),
    ("ydf_v3000", "oof_ydf_v3000.npy", "pred_ydf_v3000.npy"),
    ("xgb_v3102", "oof_xgb_v3102.npy", "pred_xgb_v3102.npy"),
    ("xgb_v3200", "oof_xgb_v3200.npy", "pred_xgb_v3200.npy"),
    ("xgb_v3201", "oof_xgb_v3201.npy", "pred_xgb_v3201.npy"),
    ("lgbm_v3300", "oof_lgbm_v3300.npy", "pred_lgbm_v3300.npy"),
    ("lgbm_v3301", "oof_lgbm_v3301.npy", "pred_lgbm_v3301.npy"),
    ("lgbm_v3302", "oof_lgbm_v3302.npy", "pred_lgbm_v3302.npy"),
    ("lgbm_v3303", "oof_lgbm_v3303.npy", "pred_lgbm_v3303.npy"),
    ("lgbm_v3304", "oof_lgbm_v3304.npy", "pred_lgbm_v3304.npy"),
    ("lgbm_v3305", "oof_lgbm_v3305.npy", "pred_lgbm_v3305.npy"),
    ("lgbm_v3307", "oof_lgbm_v3307.npy", "pred_lgbm_v3307.npy"),
    ("lgbm_v3308", "oof_lgbm_v3308.npy", "pred_lgbm_v3308.npy"),
    ("xgb_v3500", "oof_xgb_v3500.npy", "pred_xgb_v3500.npy"),
    ("xgb_v3502", "oof_xgb_v3502.npy", "pred_xgb_v3502.npy"),
    ("xgb_v3503", "oof_xgb_v3503.npy", "pred_xgb_v3503.npy"),
    ("xgb_v3504", "oof_xgb_v3504.npy", "pred_xgb_v3504.npy"),
    ("xgb_v3600", "oof_xgb_v3600.npy", "pred_xgb_v3600.npy"),
    ("lgbm_v3700", "oof_lgbm_v3700.npy", "pred_lgbm_v3700.npy"),
    ("lgbm_v3800", "oof_lgbm_v3800.npy", "pred_lgbm_v3800.npy"),
    ("ftt_v4300", "oof_ftt_v4300.npy", "pred_ftt_v4300.npy"),
    ("ftt_v4301", "oof_ftt_v4301.npy", "pred_ftt_v4301.npy"),
    ("ftt_v4302", "oof_ftt_v4302.npy", "pred_ftt_v4302.npy"),
    ("nn_v4400", "oof_nn_v4400.npy", "pred_nn_v4400.npy"),
    ("nn_v4401", "oof_nn_v4401.npy", "pred_nn_v4401.npy"),
    ("cat_v4500", "oof_cat_v4500.npy", "pred_cat_v4500.npy"),
    ("xgb_v4600", "oof_xgb_v4600.npy", "pred_xgb_v4600.npy"),
    ("realmlp_v4700", "oof_realmlp_v4700.npy", "pred_realmlp_v4700.npy"),
    ("lgbm_v4800", "oof_lgbm_v4800.npy", "pred_lgbm_v4800.npy"),
    ("tabnet_v5200", "oof_tabnet_v5200.npy", "pred_tabnet_v5200.npy"),
    ("xgb_v5300", "oof_xgb_v5300.npy", "pred_xgb_v5300.npy"),
    ("xgb_v5400", "oof_xgb_v5400.npy", "pred_xgb_v5400.npy"),
    ("xgb_v5401", "oof_xgb_v5401.npy", "pred_xgb_v5401.npy"),
    ("realmlp_v5500", "oof_realmlp_v5500.npy", "pred_realmlp_v5500.npy"),
    ("xgb_v5600", "oof_xgb_v5600.npy", "pred_xgb_v5600.npy"),
    ("cat_v5700", "oof_cat_v5700.npy", "pred_cat_v5700.npy"),
    ("lgbm_v5800", "oof_lgbm_v5800.npy", "pred_lgbm_v5800.npy"),
    ("tabm_v6400", "oof_tabm_v6400.npy", "pred_tabm_v6400.npy"),
    ("ftt_v6500", "oof_ftt_v6500.npy", "pred_ftt_v6500.npy"),
    ("realmlp_v6600", "oof_realmlp_v6600.npy", "pred_realmlp_v6600.npy"),
    ("cat_v6800", "oof_cat_v6800.npy", "pred_cat_v6800.npy"),
    ("cat_v6901", "oof_cat_v6901.npy", "pred_cat_v6901.npy"),
    ("logreg_v7011", "oof_logreg_v7011.npy", "pred_logreg_v7011.npy"),
    ("xgb_v7105", "oof_xgb_v7105.npy", "pred_xgb_v7105.npy"),
    ("xgb_v7111", "oof_xgb_v7111.npy", "pred_xgb_v7111.npy"),
    ("xgb_v7120", "oof_xgb_v7120.npy", "pred_xgb_v7120.npy"),
    ("xgb_v7123", "oof_xgb_v7123.npy", "pred_xgb_v7123.npy"),
    ("xgb_v7124", "oof_xgb_v7124.npy", "pred_xgb_v7124.npy"),
    ("tabtran_v5", "oof_tabtran_v5.npy", "pred_tabtran_v5.npy"),
    ("tabtran_v3", "oof_tabtran_v3.npy", "pred_tabtran_v3.npy"),
    ("lgbm_v7200", "oof_lgbm_v7200.npy", "pred_lgbm_v7200.npy"),
    ("cat_v7300", "oof_cat_v7300.npy", "pred_cat_v7300.npy"),
    ("realmlp_v7400", "oof_realmlp_v7400.npy", "pred_realmlp_v7400.npy"),
    ("fm_v7700", "oof_fm_v7700.npy", "pred_fm_v7700.npy"),
    ("ffm_v7800", "oof_ffm_v7800.npy", "pred_ffm_v7800.npy"),
    ("realmlp_v8100", "oof_realmlp_v8100.npy", "pred_realmlp_v8100.npy"),
    ("tabtran_v8400", "oof_tabtran_v8400.npy", "pred_tabtran_v8400.npy"),
    ("tabtran_v8401", "oof_tabtran_v8401.npy", "pred_tabtran_v8401.npy"),
    ("tabtran_v8402", "oof_tabtran_v8402.npy", "pred_tabtran_v8402.npy"),
    ("tabtran_v8403", "oof_tabtran_v8403.npy", "pred_tabtran_v8403.npy"),
    ("xgb_v8600", "oof_xgb_v8600.npy", "pred_xgb_v8600.npy"),
    ("xgb_v8601", "oof_xgb_v8601.npy", "pred_xgb_v8601.npy"),
    ("tabicl_v9500", "oof_tabicl_v9500.npy", "pred_tabicl_v9500.npy"),
    ("xgb_v8906", "oof_xgb_v8906.npy", "pred_xgb_v8906.npy"),
    ("realmlp_v10700", "oof_realmlp_v10700.npy", "pred_realmlp_v10700.npy"),
    ("tabm_v9100", "oof_tabm_v9100.npy", "pred_tabm_v9100.npy"),
    ("cat_v9300", "oof_cat_v9300.npy", "pred_cat_v9300.npy"),
    ("logit_v156_stk", "oof_logit_v156.npy", "pred_logit_v156.npy"),
    ("cat_v9400", "oof_cat_v9400.npy", "pred_cat_v9400.npy"),
    ("nn_v9401", "oof_nn_v9401.npy", "pred_nn_v9401.npy"),
    ("lgbm_v9600", "oof_lgbm_v9600.npy", "pred_lgbm_v9600.npy"),
    ("nn_v9702", "oof_nn_v9702.npy", "pred_nn_v9702.npy"),
    ("nn_v9703", "oof_nn_v9703.npy", "pred_nn_v9703.npy"),
    ("deepfm_v9704", "oof_deepfm_v9704.npy", "pred_deepfm_v9704.npy"),
    ("rff_v9705", "oof_rff_v9705.npy", "pred_rff_v9705.npy"),
    ("dcn_v9706", "oof_dcn_v9706.npy", "pred_dcn_v9706.npy"),
    ("nam_v9707", "oof_nam_v9707.npy", "pred_nam_v9707.npy"),
    ("snn_v9708", "oof_snn_v9708.npy", "pred_snn_v9708.npy"),
    ("ifan_v9709", "oof_ifan_v9709.npy", "pred_ifan_v9709.npy"),
    ("nn_v9711", "oof_nn_v9711.npy", "pred_nn_v9711.npy"),
    ("lgbm_v9802", "oof_lgbm_v9802.npy", "pred_lgbm_v9802.npy"),
    ("xgb_v9803", "oof_xgb_v9803.npy", "pred_xgb_v9803.npy"),
    ("cat_v9805", "oof_cat_v9805.npy", "pred_cat_v9805.npy"),
    ("logit_v162_stk", "oof_logit_v162.npy", "pred_logit_v162.npy"),
    ("gnn_v1911n2_stk", "oof_gnn_v1911n2.npy", "pred_gnn_v1911n2.npy"),
    ("gnn_v1918n2_stk", "oof_gnn_v1918n2.npy", "pred_gnn_v1918n2.npy"),
    ("nn_v315n2_stk", "oof_nn_v315n2.npy", "pred_nn_v315n2.npy"),
    ("nn_v315n3_stk", "oof_nn_v315n3.npy", "pred_nn_v315n3.npy"),
    ("logreg_v7014n2_stk", "oof_logreg_v7014n2.npy", "pred_logreg_v7014n2.npy"),
    ("logreg_v9200_stk", "oof_logreg_v9200.npy", "pred_logreg_v9200.npy"),
    ("realmlp_v1701n2_stk", "oof_realmlp_v1701n2.npy", "pred_realmlp_v1701n2.npy"),
    ("tabtran_v4_stk", "oof_tabtran_v4.npy", "pred_tabtran_v4.npy"),
    ("logit_v124_stk", "oof_logit_v124.npy", "pred_logit_v124.npy"),
    ("tabm_v9102_stk", "oof_tabm_v9102.npy", "pred_tabm_v9102.npy"),
    ("tabicl_v9506_stk", "oof_tabicl_v9506.npy", "pred_tabicl_v9506.npy"),
    ("saint_v9909_stk", "oof_saint_v9909.npy", "pred_saint_v9909.npy"),
    ("lgbm_v10000", "oof_lgbm_v10000.npy", "pred_lgbm_v10000.npy"),
    ("ftt_v9907", "oof_ftt_v9907.npy", "pred_ftt_v9907.npy"),
    ("cat_v10300", "oof_cat_v10300.npy", "pred_cat_v10300.npy"),
    ("cat_v10311", "oof_cat_v10311.npy", "pred_cat_v10311.npy"),
    ("cat_v10313", "oof_cat_v10313.npy", "pred_cat_v10313.npy"),
    ("cat_v10315", "oof_cat_v10315.npy", "pred_cat_v10315.npy"),
    ("cat_v10325", "oof_cat_v10325.npy", "pred_cat_v10325.npy"),
    ("cat_v10326", "oof_cat_v10326.npy", "pred_cat_v10326.npy"),
    ("cat_v10327", "oof_cat_v10327.npy", "pred_cat_v10327.npy"),
    ("cat_v10328", "oof_cat_v10328.npy", "pred_cat_v10328.npy"),
    ("cox_v8007", "oof_cox_v8007.npy", "pred_cox_v8007.npy"),
    ("rf_v10500", "oof_rf_v10500.npy", "pred_rf_v10500.npy"),
    ("realmlp_v10900", "oof_realmlp_v10900.npy", "pred_realmlp_v10900.npy"),
    ("xgb_v5409_stk", "oof_xgb_v5409.npy", "pred_xgb_v5409.npy"),
    ("xgb_v3614_stk", "oof_xgb_v3614.npy", "pred_xgb_v3614.npy"),
    ("cat_v10324_stk", "oof_cat_v10324.npy", "pred_cat_v10324.npy"),
]

# -----------------------------
# Helpers
# -----------------------------
def to_binary_y(train_df, target_col):
    y = train_df[target_col]
    if y.dtype == "object" or str(y.dtype).startswith("string"):
        y = y.map({"No": 0, "Yes": 1})
    y = pd.to_numeric(y, errors="coerce")
    if y.isna().any():
        bad = train_df.loc[y.isna(), target_col].astype(str).value_counts().head(10)
        raise ValueError(
            f"Could not convert some target values in '{target_col}' to 0/1. "
            f"Top unparsed values:\n{bad}"
        )
    y = y.astype(np.int32).to_numpy()
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f"Target after conversion is not binary 0/1. Found: {np.unique(y)[:20]}")
    return y

def auc_cpu(y_true, scores):
    return float(roc_auc_score(y_true, scores))

def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def prob_to_logit(p, eps=1e-15, clip=30.0):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, 1.0 - eps)
    z = np.log(p / (1.0 - p))
    return np.clip(z, -clip, clip)

def load_preds_1d(path: str, expected_len: int, col_for_csv: str = "Churn", train_truncate=False) -> np.ndarray:
    path = f"{NUMPY_PATH}/{path}"
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
        if col_for_csv not in df.columns:
            raise ValueError(
                f"CSV '{path}' does not contain column '{col_for_csv}'. "
                f"Available columns: {list(df.columns)[:30]}"
            )
        arr = df[col_for_csv].to_numpy()
    else:
        arr = np.load(path)

    arr = np.asarray(arr, dtype=np.float64)

    if arr.ndim == 2:
        if arr.shape[1] == 1:
            arr = arr.reshape(-1)
        elif arr.shape[0] == expected_len:
            arr = arr.mean(axis=1)
        else:
            raise ValueError(f"2D array '{path}' has unexpected shape {arr.shape} for expected_len={expected_len}")

    arr = arr.reshape(-1)
    if train_truncate:
        arr = arr[:594194]

    if arr.shape[0] != expected_len:
        raise ValueError(f"Length mismatch for '{path}': got {arr.shape[0]} vs expected {expected_len}")
    return arr

def ordinal_common_fold_ranks(values, common_size):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    n = values.size
    if n == 0:
        return np.empty(0, dtype=np.int32)
    if common_size <= 1:
        return np.zeros(n, dtype=np.int32)
    order = np.argsort(values, kind="mergesort")
    pos = np.arange(n, dtype=np.int64)
    ranks_sorted = np.clip((pos * common_size) // n, 0, common_size - 1).astype(np.int32)
    ranks = np.empty(n, dtype=np.int32)
    ranks[order] = ranks_sorted
    return ranks

def fold_calibrate_oof_probs(oof, folds, eps=1e-15):
    oof = np.asarray(oof, dtype=np.float64).reshape(-1)
    min_fold_size = int(min(len(va_idx) for _, va_idx in folds))
    ranks = np.empty(oof.shape[0], dtype=np.int32)
    for _, va_idx in folds:
        ranks[va_idx] = ordinal_common_fold_ranks(oof[va_idx], common_size=min_fold_size)
    rank_means = pd.DataFrame({"rank": ranks, "prob": oof}).groupby("rank", sort=True)["prob"].mean()
    calibrated = rank_means.iloc[ranks].to_numpy(dtype=np.float64)
    return np.clip(calibrated, eps, 1.0 - eps)

def ensure_numpy(x, dtype=None):
    if isinstance(x, np.ndarray):
        arr = x
    elif hasattr(x, "to_numpy"):
        arr = x.to_numpy()
    elif hasattr(x, "get"):
        arr = x.get()
    elif hasattr(x, "values_host"):
        arr = x.values_host
    else:
        arr = np.asarray(x)
    if dtype is not None:
        arr = arr.astype(dtype, copy=False)
    return arr

def make_meta_model():
    kwargs = dict(
        penalty=META_PENALTY,
        C=META_C,
        fit_intercept=FIT_INTERCEPT,
        max_iter=META_MAX_ITER,
        tol=META_TOL,
        solver=META_SOLVER,
        output_type="numpy",
        verbose=False,
    )
    if META_PENALTY == "elasticnet":
        kwargs["l1_ratio"] = META_L1_RATIO
    return cuLogisticRegression(**kwargs)

# ============================================================
# 1) LOAD TRAIN/TEST + TARGET + FOLDS
# ============================================================
print("Loading:", TRAIN_CSV)
train = pd.read_csv(TRAIN_CSV)
print("Loading:", TEST_CSV)
test = pd.read_csv(TEST_CSV)

y = to_binary_y(train, TARGET_COL)
N = len(y)
M = len(test)
print(f"Train rows: {N} | Pos rate: {y.mean().round(5)}")
print(f"Test rows : {M}")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(np.zeros(N, dtype=np.int8), y))
print(f"Built {N_SPLITS}-fold StratifiedKFold (seed={RANDOM_STATE})")

# ============================================================
# 2) LOAD OOF + TEST FILES
# ============================================================
names        = []
x_train_prob = []
x_test_prob  = []

print(f"\nLoading {len(FILES)} OOF + TEST file pairs...")
for name, oof_path, pred_path in FILES:
    oof  = load_preds_1d(oof_path,  expected_len=N, col_for_csv="Churn", train_truncate=True)
    pred = load_preds_1d(pred_path, expected_len=M, col_for_csv="Churn")

    oof  = np.clip(oof,  EPS, 1.0 - EPS)
    pred = np.clip(pred, EPS, 1.0 - EPS)

    if FOLD_CALIBRATE:
        oof = fold_calibrate_oof_probs(oof, FOLDS, eps=EPS)

    names.append(name)
    x_train_prob.append(oof)
    x_test_prob.append(pred)

x_train_prob = np.stack(x_train_prob, axis=1)
x_test_prob  = np.stack(x_test_prob,  axis=1)
print(f"OOF matrix : {x_train_prob.shape}")
print(f"TEST matrix: {x_test_prob.shape}")

# ============================================================
# 3) PROB -> LOGIT FEATURES
# ============================================================
X_train = prob_to_logit(x_train_prob, eps=EPS, clip=LOGIT_CLIP).astype(np.float32)
X_test  = prob_to_logit(x_test_prob,  eps=EPS, clip=LOGIT_CLIP).astype(np.float32)

# ============================================================
# 4) FOLD-WISE cuML LOGISTIC REGRESSION (HONEST OOF AUC)
# ============================================================
meta_oof        = np.zeros(N, dtype=np.float64)
meta_test_folds = np.zeros((M, N_SPLITS), dtype=np.float64)

print("\nTraining fold-wise cuML logistic regression stacker...")
for fold, (tr_idx, va_idx) in enumerate(FOLDS, 1):
    X_tr, y_tr = X_train[tr_idx], y[tr_idx].astype(np.int32, copy=False)
    X_va        = X_train[va_idx]
    y_va        = y[va_idx].astype(np.int32, copy=False)

    clf   = make_meta_model()
    clf.fit(X_tr, y_tr)

    va_pred = ensure_numpy(clf.predict_proba(X_va))[:, 1].astype(np.float64)
    te_pred = ensure_numpy(clf.predict_proba(X_test))[:, 1].astype(np.float64)

    meta_oof[va_idx]          = va_pred
    meta_test_folds[:, fold-1] = te_pred

    coef_fold = ensure_numpy(clf.coef_).reshape(-1)
    print(
        f"Fold {fold}/{N_SPLITS} | AUC={auc_cpu(y_va, va_pred):.6f} | "
        f"nonzero_coef={int(np.sum(np.abs(coef_fold) > 1e-12))}"
    )
    del X_tr, y_tr, X_va, y_va, clf, va_pred, te_pred, coef_fold
    gc.collect()

cv_auc = auc_cpu(y, meta_oof)
print(f"\ncuML logistic stack OOF AUC: {cv_auc:.6f}")

# ============================================================
# 5) FIT FINAL MODEL ON FULL DATA
# ============================================================
final_test_preds = []
for k in range(10):
    print(f"Full-data fit {k+1}/10 ...")
    final_clf = make_meta_model()
    final_clf.fit(X_train, y.astype(np.int32, copy=False))
    final_test_preds.append(ensure_numpy(final_clf.predict_proba(X_test))[:, 1].astype(np.float64))

final_test_pred = np.mean(final_test_preds, axis=0)
full_fit_auc = auc_cpu(y, ensure_numpy(final_clf.predict_proba(X_train))[:, 1])
print(f"Full-fit train AUC (optimistic): {full_fit_auc:.6f}")

# ============================================================
# 6) SAVE OOF + TEST PREDS
# ============================================================
np.save(OOF_PATH,  meta_oof)
np.save(TEST_PATH, final_test_pred)
print(f"Saved OOF : {OOF_PATH}")
print(f"Saved TEST: {TEST_PATH}")

# ============================================================
# 7) COEFFICIENTS / WEIGHTS TABLE
# ============================================================
coefs = ensure_numpy(final_clf.coef_).reshape(-1).astype(np.float64)
intercept_val = float(ensure_numpy(final_clf.intercept_).reshape(-1)[0]) if FIT_INTERCEPT else 0.0

dfw = pd.DataFrame({
    "model":    names,
    "coef":     coefs,
    "abs_coef": np.abs(coefs),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("\nFinal cuML logistic regression coefficients (full fit):")
print(dfw.to_string(index=False))
print(f"\nIntercept: {intercept_val}")

dfw.to_csv(WEIGHTS_PATH, index=False)
print(f"Saved weights: {WEIGHTS_PATH}")

# Sanity check: manual rebuild from saved coefs
coef_vec  = dfw.set_index("model").loc[names, "coef"].to_numpy(dtype=np.float64)
manual_auc = auc_cpu(y, sigmoid(X_train.astype(np.float64) @ coef_vec + intercept_val))
print(f"Sanity full-fit AUC from saved coefs: {manual_auc:.6f}")

# ============================================================
# 8) WRITE SUBMISSION
# ============================================================
if ID_COL in test.columns:
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: final_test_pred})
else:
    sub = pd.DataFrame({TARGET_COL: final_test_pred})

sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved submission: {SUBMISSION_PATH}")
print(sub.head())

print(f"\n{'='*50}")
print(f"Base models           : {len(names)}")
print(f"OOF AUC (honest CV)   : {cv_auc:.6f}")
print(f"Full-fit train AUC    : {full_fit_auc:.6f}")
print(f"FOLD_CALIBRATE        : {FOLD_CALIBRATE}")
print(f"META_C / PENALTY      : {META_C} / {META_PENALTY}")